# Parallel Tool Calls, Errors & Partial Failure

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 12 §12.5 · type: walkthrough*

A model can ask for **several** tools in one turn. In 12-02 your loop already
collected every call and returned every result — but it ran them one after another.
Here you'll run the independent ones **concurrently** with `asyncio`, and, more
importantly, keep the agent alive when one of them **fails**. Partial failure is
the normal case, not the edge case.

## 🧠 Why this matters

Asked to "compare our refund policy with the cancellation policy," a model emits
*two* `read_file` calls in a single response. Run them sequentially and the user
waits for the sum of both latencies; run them concurrently and they wait for the
slowest one. Once tools do real I/O, that difference is the whole user experience.

But concurrency multiplies the ways things break. When three parallel calls run and
one times out, you do **not** abort the turn. You return two successes and one
`is_error` result, and let the model decide: retry, work with what it has, or tell
the user honestly what it couldn't check. Two rules make this safe:

- **You must still send a `tool_result` for every `tool_use_id`** — even a failed
  one — or the API rejects your next request.
- **Only parallelize what's safe to parallelize**: reads, yes; two writes to the
  same record, no.

## Objectives & prerequisites

**By the end you can:**

- Inspect a turn with **multiple** `tool_use` blocks.
- Execute independent tools **concurrently** with `asyncio.gather`, preserving each
  call's id.
- Turn a tool failure into a **structured `is_error` result** instead of a crash,
  and watch the model recover.
- Judge **when parallelism helps** and when ordering/consistency forbids it.

**Prereqs:** 12-02 (the loop, the dispatcher, `is_error` results); async basics
from Chapter 4 (`learn/part-02-.../04-*`).

**Run first:** the Setup cell. Defaults to `MOCK=1` — fully offline and free.

In [ ]:
# --- Setup -------------------------------------------------------------------
import asyncio
import json
import os
import random
import time

from dotenv import load_dotenv

load_dotenv()  # never hardcode keys

# MOCK=1 (default): a scripted multi-tool model + simulated I/O latency, so the
# loop runs offline and deterministically. MOCK=0 hits the live API (one
# multi-tool run).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(12)  # determinism for anything stochastic

MODEL = "claude-opus-4-8"

if MOCK:
    print("MOCK mode: scripted multi-tool model. No API key, nothing billed.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Set it in your environment/.env or use COMPANION_MOCK=1."
        )
    import anthropic

    client = anthropic.Anthropic()
    print(f"LIVE mode: calling {MODEL}. One multi-tool run will be billed.")

## 1 · The tools — two reads and one that deliberately fails

Three tools. Two read policy docs (with a small simulated delay so concurrency is
visible). The third — `check_live_status` — **always fails**, standing in for the
flaky network dependency every real system has. The dispatcher catches it and
returns a structured error, exactly as in 12-02.

In [ ]:
POLICIES = {
    "refund": "Full refund within 30 days of purchase, to the original method.",
    "cancellation": "Cancel anytime; access lasts until the period ends. No fee.",
}


def read_policy(topic: str) -> str:
    """A read tool with a small simulated latency (so parallelism is visible)."""
    time.sleep(0.05)  # pretend this is disk/DB I/O
    if topic not in POLICIES:
        raise ValueError(f"unknown topic {topic!r}; valid: {list(POLICIES)}")
    return POLICIES[topic]


def check_live_status(service: str) -> str:
    """A tool that ALWAYS fails -- our stand-in for a flaky dependency."""
    time.sleep(0.05)
    raise ConnectionError(f"status endpoint for {service!r} timed out")


REGISTRY = {"read_policy": read_policy, "check_live_status": check_live_status}


def run_tool(name: str, args: dict) -> tuple[str, bool]:
    """Execute one tool; return (result, is_error). NEVER raises -- a failed call
    still owes the model a tool_result, or the next API request is rejected."""
    fn = REGISTRY.get(name)
    if fn is None:
        return f"Unknown tool: {name!r}", True
    try:
        return fn(**args), False
    except Exception as exc:
        # Phrase the error FOR THE MODEL so it can adapt on the next turn.
        return f"{type(exc).__name__}: {exc}", True


print(run_tool("read_policy", {"topic": "refund"}))
print(run_tool("check_live_status", {"service": "billing"}))

## 2 · A turn with *several* `tool_use` blocks

When the user asks to compare two policies *and* check live status, the model
returns three `tool_use` blocks in one response. Here's that canned turn —
inspect the shape: one assistant message, a list of three independent requests,
each with its own `id`.

In [ ]:
multi_tool_turn = {
    "stop_reason": "tool_use",
    "content": [
        {"type": "text", "text": "I'll check both policies and the live status."},
        {"type": "tool_use", "id": "toolu_a",
         "name": "read_policy", "input": {"topic": "refund"}},
        {"type": "tool_use", "id": "toolu_b",
         "name": "read_policy", "input": {"topic": "cancellation"}},
        {"type": "tool_use", "id": "toolu_c",
         "name": "check_live_status", "input": {"service": "billing"}},
    ],
}

tool_use_blocks = [b for b in multi_tool_turn["content"] if b["type"] == "tool_use"]
print(f"{len(tool_use_blocks)} tool calls in one turn:")
for b in tool_use_blocks:
    print(f"  id={b['id']}  {b['name']}({b['input']})")

## 3 · Run them concurrently — and preserve every id

`asyncio.gather` fans the calls out at once. Each is wrapped in
`asyncio.wait_for(..., timeout)` so a hung tool can't hang the turn, and the sync
tools run via `asyncio.to_thread` to stay off the event loop. Crucially, **a
timeout or crash still produces a `tool_result`** carrying the original
`tool_use_id` — partial failure, fully accounted for.

In [ ]:
async def run_tools_parallel(blocks, timeout=20.0):
    """Execute tool_use blocks concurrently; return one tool_result per block.

    Independent calls run at once. A timeout or exception becomes an is_error
    result (never an unhandled raise), and every result carries its tool_use_id.
    """
    async def one(block):
        try:
            output, is_error = await asyncio.wait_for(
                asyncio.to_thread(run_tool, block["name"], block["input"]),
                timeout=timeout,
            )
        except asyncio.TimeoutError:
            output, is_error = (
                f"Tool {block['name']} timed out after {timeout}s; try again "
                f"or proceed without it.",
                True,
            )
        return {
            "type": "tool_result",
            "tool_use_id": block["id"],   # preserve the id -> matches the request
            "content": output,
            "is_error": is_error,
        }

    return await asyncio.gather(*(one(b) for b in blocks))

## 4 · 🔮 Predict: what comes back when one of three fails?

We'll run all three calls concurrently. Two reads succeed; `check_live_status`
raises a `ConnectionError`.

**Predict before running:** How many `tool_result`s come back? What is `is_error`
on each? Does the failure of the third call stop the first two from returning? And
does the wall-clock time look more like *one* tool's latency or the *sum* of three?

In [ ]:
start = time.perf_counter()
results = await run_tools_parallel(tool_use_blocks)
elapsed = time.perf_counter() - start

for r in results:
    flag = "ERROR" if r["is_error"] else "ok   "
    print(f"[{flag}] {r['tool_use_id']}: {r['content']}")

print(f"\n3 tools, ~0.05s each. Wall time: {elapsed:.2f}s "
      f"(concurrent ~= one call, not the sum).")

**What you just saw.** Three results came back — one per `tool_use_id`. Two are
clean; the third is `is_error: True` with a readable message. The failure was
*isolated*: it didn't touch the successful reads, and it didn't raise out of the
loop. And the wall-clock time is close to a single call's latency, not three —
that's the concurrency payoff.

## 5 · ⚠️ Pitfall: dropping the result of a failed call

The tempting "fix" when a tool throws is to skip it — `continue` past the failure
and send back only the successes. That **breaks the next API request**: the
provider requires exactly one `tool_result` for every `tool_use_id` it sent. Miss
one and the call is rejected with a confusing error far from the real cause.

In [ ]:
def broken_collect(blocks):
    """WRONG: silently drops failed calls -> a missing tool_result."""
    out = []
    for b in blocks:
        output, is_error = run_tool(b["name"], b["input"])
        if is_error:
            continue          # <-- the bug: this id never gets a result
        out.append({"type": "tool_result", "tool_use_id": b["id"], "content": output})
    return out


sent_ids = {b["id"] for b in tool_use_blocks}
returned_ids = {r["tool_use_id"] for r in broken_collect(tool_use_blocks)}
missing = sent_ids - returned_ids

print("ids the model asked about :", sorted(sent_ids))
print("ids we returned a result for:", sorted(returned_ids))
print("MISSING (API will reject!) :", sorted(missing))
# The correct version (run_tools_parallel) returns a result for EVERY id, marking
# failures with is_error instead of dropping them.

## 6 · The model recovers from a partial failure

Feed the three results back. The two successes give the model what it needs to
answer the comparison; the one error tells it, in plain language, what it couldn't
check. A capable model **acknowledges the gap and answers anyway** rather than
giving up. Here's the canned follow-up turn that mirrors that behavior.

In [ ]:
def model_followup(results):
    """Canned final turn: the model uses the successes and notes the failure.

    Live, you'd append {'role':'assistant', 'content': turn_content} then
    {'role':'user', 'content': results} and call the API again; it produces text
    like this on its own.
    """
    had_error = any(r["is_error"] for r in results)
    note = (" I couldn't confirm live billing status (the status check timed "
            "out), but the policies themselves are:") if had_error else ""
    return {
        "stop_reason": "end_turn",
        "content": [{
            "type": "text",
            "text": (
                f"Here's the comparison.{note} Refunds: "
                f"{POLICIES['refund']} Cancellation: {POLICIES['cancellation']}"
            ),
        }],
    }


final = model_followup(results)
print("stop_reason:", final["stop_reason"])
print(final["content"][0]["text"])

A few hard-won rules round this out. Write error messages **for the model**, not
for a stack-trace reader: "date must be ISO 8601, e.g. 2026-06-10" earns a
corrected retry; a bare `ValueError` earns a guess. **Cap retries** — if the model
has failed the same tool three times, return an error that says so and tells it to
stop, or you'll pay for an expensive loop of optimism. And **log every call and
result** with the request id: the tool transcript is your flight recorder, and
Chapter 22's evals replay exactly these transcripts.

## 🎯 Senior lens: when parallelism helps, and when it's forbidden

Concurrency is free latency *only* for operations that don't interfere. The split
follows side effects, again. **Reads** — policy lookups, searches, GETs — are
independent: fan them out. **Writes** that touch the same resource are not: two
concurrent `issue_refund` calls on one invoice is a race that can double-refund.
When ordering or consistency matters, force sequential execution (both major APIs
let you disable parallel tool use — Anthropic via `disable_parallel_tool_use` on
`tool_choice`), but the better fix is usually tools whose semantics don't depend on
order at all. The judgment a senior brings isn't "parallelize everything"; it's
knowing which calls are *safe* to overlap and designing the rest so the question
doesn't arise.

## Recap

- A single turn can carry **multiple `tool_use` blocks**; your loop must return a
  result for **every** one.
- `asyncio.gather` + `asyncio.wait_for` runs independent tools **concurrently**
  with per-call **timeouts**, collapsing latency to roughly the slowest call.
- **Partial failure is normal:** a timeout or crash becomes an `is_error` result
  carrying its `tool_use_id`; the other calls are unaffected.
- **Never drop a failed call's result** — a missing `tool_result` gets your next
  API request rejected.
- **Parallelize reads; serialize conflicting writes.** Recovery is a dialogue kept
  alive with informative, model-readable errors.

## Exercises

Use the empty cells below. (Solutions arrive in `solutions/` in Phase 2.)

1. **Force a timeout.** Lower `run_tools_parallel`'s `timeout` to `0.01` so the
   `time.sleep(0.05)` tools exceed it. Predict how many results come back, and what
   `is_error` and `content` say for each, before you run it.
2. **Sequential vs concurrent.** Write a `run_tools_sequential` that awaits each
   tool in turn and time both on the three calls. Predict the ratio of wall-clock
   times, then measure — does it match your mental model of the latencies?
3. **A second write hazard.** Add an `issue_refund(invoice_id)` write tool and have
   the model emit two calls for the *same* `invoice_id` in one turn. Explain why
   `run_tools_parallel` is the wrong executor here, and sketch the guard you'd add.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

## Next

You can now run tools in parallel and keep an agent alive through partial failure —
the last piece of the raw loop. Time to compare your hand-built engine to the
production one.

- 🔧 **Blueprint (the production loop):**
  [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/) — typed
  tools, concurrency, retries with caps, and telemetry hooks. You built the toy
  across 12-01 → 12-03; that's the real one.
- 🧩 **Template:** the tool-definition pattern feeds
  [`../../../templates/agent-project-starter/`](../../../templates/agent-project-starter/).
- 🎓 **Capstone:** advances `capstone/agents/raw/` and `capstone/agents/tools/`;
  checkpoint `checkpoints/ch12-tool-loop`.
- 📖 **Book:** §12.5 (parallel calls, error recovery, partial failures). The
  loop you built reappears in LangGraph and Pydantic AI in Part VII — now you have a
  baseline you fully understand.